# Telemetry Collector — Demo (Tiếng Việt | English)

Tiếng Việt:
Notebook nhỏ này kết nối tới WebSocket của Pidron, thu các message telemetry
 và lưu vào file `telemetry_capture.json` để phân tích sau.

English:
This small notebook connects to the Pidron WebSocket endpoint, captures
 telemetry messages and saves them to `telemetry_capture.json` for later analysis.

In [ ]:
# Install dependencies (run once)
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'websocket-client', 'pandas', 'matplotlib'])

In [ ]:
# Configuration: default WebSocket endpoint used by Pidron is /api/ws
WS_URL = 'ws://127.0.0.1:8080/api/ws'  # server WebSocket endpoint
DURATION_SECONDS = 20  # how long to collect telemetry
OUTPUT_FILE = 'telemetry_capture.json'

from websocket import create_connection, WebSocketTimeoutException
import json, time

print('Connecting to', WS_URL)
try:
    ws = create_connection(WS_URL, timeout=5)
except Exception as e:
    raise SystemExit(f'Failed to connect to {WS_URL}: {e}')

start = time.time()
collected = []
print('Collecting telemetry for', DURATION_SECONDS, 'seconds...')
while time.time() - start < DURATION_SECONDS:
    try:
        msg = ws.recv()
    except WebSocketTimeoutException:
        continue
    except Exception as e:
        print('Recv error:', e)
        break
    if not msg:
        continue
    try:
        obj = json.loads(msg)
    except Exception:
        # not JSON (may be ping/other)
        continue
    # store only telemetry messages to reduce file size
    if obj.get('type') == 'telemetry':
        ts = time.time()
        collected.append({'t': ts, 'payload': obj})

# save to file
with open(OUTPUT_FILE, 'w') as f:
    json.dump(collected, f)

print('Saved', len(collected), 'telemetry entries to', OUTPUT_FILE)
ws.close()

In [ ]:
# Load captured telemetry and plot drone positions over time for one drone
import json, pandas as pd, matplotlib.pyplot as plt

INPUT_FILE = 'telemetry_capture.json'
with open(INPUT_FILE, 'r') as f:
    records = json.load(f)

# flatten into rows for a selected drone id
rows = []
for rec in records:
    t = rec['t']
    payload = rec['payload']
    drones = payload.get('drones', {})
    for drone_id, info in drones.items():
        telemetry = info.get('telemetry', {})
        pos = telemetry.get('pos') or telemetry.get('position') or [None, None, None]
        rows.append({'t': t, 'drone': drone_id, 'x': pos[0], 'y': pos[1], 'z': pos[2], 'battery': telemetry.get('battery')})

df = pd.DataFrame(rows)
if df.empty:
    print('No telemetry rows found in', INPUT_FILE)
else:
    print('Captured drones:', df['drone'].unique())
    # pick first drone
    sel = df['drone'].unique()[0]
    ddf = df[df['drone'] == sel].sort_values('t')
    fig, ax = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
    ax[0].plot(ddf['t'], ddf['x'], label='x')
    ax[0].set_ylabel('X (m)')
    ax[1].plot(ddf['t'], ddf['y'], label='y', color='orange')
    ax[1].set_ylabel('Y (m)')
    ax[2].plot(ddf['t'], ddf['z'], label='z', color='green')
    ax[2].set_ylabel('Z (m)')
    ax[2].set_xlabel('time (s)')
    fig.suptitle(f'Drone trajectory over time: {sel}')
    plt.tight_layout()
    plt.show()

Next steps:
- Extend parsing to extract velocity, attitude and motor outputs for richer plots.
- Export selected time ranges as CSV for external analysis.
- Integrate this capture loop as a headless Python script for automated benchmarking.